# Part 2, Question 1: Campaign Team Tracking and Coverage Reconciliation

## Notebook 2: Quality Assurance (QA) of GPS track data.

### Objective

The goal of QA is to remove or flag GPS points that are implausible, unreliable, or non‑operational, while preserving valid movement patterns.
Your QA must be:

### This section loads the packages required for data manipulation, spatial analysis, visualisation and reporting. Keeping all dependencies in one place improves reproducibility and makes the analysis environment easy to recreate.

In [7]:
## let import our the necessary library for this task

import geopandas as gpd
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
gpkg_path = PROCESSED_DIR / "campaign_data.gpkg"

# Import layers
tracks_gdf = gpd.read_file(gpkg_path, layer="gps_tracks")
settlements_gdf = gpd.read_file(gpkg_path, layer="settlements")
wards = gpd.read_file(gpkg_path, layer="wards")
lgas = gpd.read_file(gpkg_path, layer="lgas")

In [8]:
# Load the raw GPS tracks layer from your GeoPackage
tracks_gdf = gpd.read_file(gpkg_path, layer="gps_tracks")

# Convert timestamp to datetime
tracks_gdf["timestamp"] = pd.to_datetime(tracks_gdf["timestamp"], errors="coerce")

# Sort by team and time to ensure correct sequence
tracks_gdf = tracks_gdf.sort_values(["team_id", "timestamp"]).reset_index(drop=True)

# Quick verification
print(tracks_gdf.shape)
tracks_gdf.head()

(956702, 9)


,team_id,logger_id,timestamp,longitude,latitude,accuracy_m,speed_kmh,track_file,geometry
0,T01,GL-1005,2026-03-09 07:10:00,7.149428,11.050412,54.6,3.26,T01_2026-03-09,POINT (7.14943 11.05041)
1,T01,GL-1005,2026-03-09 07:11:00,7.150248,11.049904,14.8,3.88,T01_2026-03-09,POINT (7.15025 11.0499)
2,T01,GL-1005,2026-03-09 07:12:00,7.150536,11.049718,39.6,3.92,T01_2026-03-09,POINT (7.15054 11.04972)
3,T01,GL-1005,2026-03-09 07:13:00,7.151675,11.049777,52.2,3.26,T01_2026-03-09,POINT (7.15168 11.04978)
4,T01,GL-1005,2026-03-09 07:14:00,7.150969,11.048901,18.3,0.48,T01_2026-03-09,POINT (7.15097 11.0489)


### Why convert timestamp to datetime?

Most raw GPS logs store time as strings (e.g. "2026-07-30 09:13:42").
As long as it’s a string, Python treats it like text, not time. 
In addition, for Requirement 2, almost all your QA rules depend on temporal logic, not text:

* Duty hours rule → We need the hour of day (08:00–17:00)

* Fix gap rule → We need the difference between consecutive timestamps

* Stationary cluster rule → We need to know if the device stayed in one place for ≥ X minutes

## Rule A — Implausible Speeds  
GPS loggers occasionally record spikes due to satellite drift or interpolation errors.
Human walking speed is typically 3–6 km/h, and motorbike movement for campaign teams rarely exceeds 40 km/h.
A maximum plausible speed threshold of 60 km/h is therefore used.
Points above this threshold are flagged as implausible and excluded from further analysis.

In [10]:
# Rule A: Implausible speeds > 60 km/h
rule_speed = tracks_gdf["speed_kmh"] > 60

print("Implausible speed points:", rule_speed.sum())


Implausible speed points: 4702


In [11]:
#we will add the flagged column (but do NOT remove points yet) this is important for transparency.
tracks_gdf["qa_speed"] = rule_speed


## Rule B — Positional Accuracy > 30 m

Justification 

##### Consumer GPS devices typically achieve 5–20 m accuracy

##### WHO field guidance uses ≤ 30 m as acceptable for campaign tracking

##### Accuracy > 30 m indicates:

* poor satellite lock

* urban multipath interference

* logger obstruction

* degraded positional reliability

Therefore, accuracy > 30 m is flagged.

In [12]:
# Rule B: Positional accuracy > 30 m
rule_accuracy = tracks_gdf["accuracy_m"] > 30

print("Poor accuracy points:", rule_accuracy.sum())

tracks_gdf["qa_accuracy"] = rule_accuracy


Poor accuracy points: 396130


Rule B (Positional accuracy > 30 m):  
396,130 points were flagged as having poor positional accuracy.
This is expected in SIA campaigns where teams operate in dense settlements, narrow streets, and areas with multipath interference.
The threshold of 30 m is based on WHO field guidance for GPS‑supported campaign monitoring, which considers 5–20 m typical and ≤30 m acceptable.
Points above this threshold are retained for analysis but flagged as lower‑confidence fixes.

## Rule C — Points outside campaign duty hours (08:00–17:00)

#### Implement Rule C — Duty‑hour QA

In [16]:
# Add the hour column
tracks_gdf["hour"] = tracks_gdf["timestamp"].dt.hour


In [15]:
rule_hours = ~tracks_gdf["hour"].between(8, 17)

print("Outside duty‑hour points:", rule_hours.sum())

tracks_gdf["qa_hours"] = rule_hours


Outside duty‑hour points: 530611


### Rule C (Outside duty hours):  
530,611 points were flagged as occurring outside the operational window of 08:00–17:00.
SIA campaigns typically operate during daylight hours, and points outside this window likely represent logger warm‑up, overnight recording, transit movement, or non‑campaign activity.
These points are flagged rather than removed, as they may still contribute to fix‑sequence diagnostics and help identify logger behaviour patterns

## Rule D — Gaps in the fix sequence (> 120 seconds)
GPS loggers normally record every 1–5 seconds This could be due to

* Gaps > 120 seconds indicate:

* logger malfunction

* battery drop

* loss of satellite lock

* team inactivity

* device obstruction

In [21]:
#Add the time difference column
tracks_gdf["time_diff"] = (
    tracks_gdf.groupby("team_id")["timestamp"]
    .diff()
    .dt.total_seconds()
)
# Now we will apply the rule 

rule_gaps = tracks_gdf["time_diff"] > 120

print("Fix‑sequence gap points:", rule_gaps.sum())

tracks_gdf["qa_gaps"] = rule_gaps


Fix‑sequence gap points: 159


### Rule D (Fix‑sequence gaps > 120 seconds):  
Only 159 points were flagged as having gaps longer than 120 seconds between consecutive fixes.
GPS loggers typically record every 1–5 seconds, and gaps longer than 120 seconds indicate logger malfunction, battery depletion, or prolonged inactivity.
The low number of flagged gaps suggests stable logger performance and consistent satellite lock throughout the campaign period.

## Rule E — Stationary Clusters


In [23]:
tracks_gdf["time_diff"]


0          NaN
1         60.0
2         60.0
3         60.0
4         60.0
          ... 
956697    60.0
956698    60.0
956699    60.0
956700    60.0
956701    60.0
Name: time_diff, Length: 956702, dtype: float64

In [25]:
# Apply the stationary rule
rule_stationary = (
    (tracks_gdf["speed_kmh"] < 0.5) &
    (tracks_gdf["time_diff"] > 300)
)

print("Stationary cluster points:", rule_stationary.sum())

tracks_gdf["qa_stationary"] = rule_stationary



Stationary cluster points: 24


## Rule E (Stationary clusters):  
Only 24 points were flagged as stationary clusters, defined as fixes where speed < 0.5 km/h for more than 300 seconds.
These points typically represent periods where the logger was left in one place, such as during breaks or charging.
The low number of stationary clusters indicates that teams maintained consistent movement and did not leave devices idle for extended periods, reducing the risk of inflated point density around non‑operational locations.

### Step 1 — Combine all QA flags

In [26]:
tracks_gdf["qa_flag"] = (
    tracks_gdf["qa_speed"] |
    tracks_gdf["qa_accuracy"] |
    tracks_gdf["qa_hours"] |
    tracks_gdf["qa_gaps"] |
    tracks_gdf["qa_stationary"]
)


### Step 2 — Create a QA summary table

In [27]:


qa_report = pd.DataFrame({
    "Rule": [
        "Implausible speed > 60 km/h",
        "Accuracy > 30 m",
        "Outside duty hours (08–17)",
        "Time gap > 120 sec",
        "Stationary cluster"
    ],
    "Flagged": [
        tracks_gdf["qa_speed"].sum(),
        tracks_gdf["qa_accuracy"].sum(),
        tracks_gdf["qa_hours"].sum(),
        tracks_gdf["qa_gaps"].sum(),
        tracks_gdf["qa_stationary"].sum()
    ]
})

qa_report


,Rule,Flagged
0,Implausible speed > 60 km/h,4702
1,Accuracy > 30 m,396130
2,Outside duty hours (08–17),530611
3,Time gap > 120 sec,159
4,Stationary cluster,24


## Step 3 — Create the cleaned dataset

In [30]:


tracks_clean = tracks_gdf[~tracks_gdf["qa_flag"]].copy()
print("Cleaned track points:", tracks_clean.shape[0])


Cleaned track points: 255673


## Step 4 — Export cleaned tracks back into your GeoPackage

In [31]:
tracks_clean.to_file(
    gpkg_path,
    layer="gps_tracks_clean",
    driver="GPKG"
)


In [32]:
import fiona

gpkg_path = PROCESSED_DIR / "campaign_data.gpkg"

print(f"Layers in GeoPackage:")
print(fiona.listlayers(gpkg_path))


Layers in GeoPackage:
['gps_tracks', 'settlements', 'wards', 'lgas', 'state', 'gps_tracks_clean']
